# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.
This includes printing dataset name and description for context.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Access the metadata as an object
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).
We'll enumerate all `RecordSet`s and their fields to identify their `@id` values for further extraction.

**Note:** All references use the entity `@id`, per Croissant specification.

In [ ]:
# The mlcroissant schema provides access to record sets via metadata.record_sets.
print('Available record sets:')
rs_overview = []
for rs in getattr(metadata, 'record_sets', []):
    print(f"- @id: {getattr(rs, '@id', '')}, name: {getattr(rs, 'name', '')}")
    rs_overview.append(getattr(rs, '@id', ''))
    if hasattr(rs, 'fields'):
        print('  Fields:')
        for field in getattr(rs, 'fields', []):
            print(f"    - @id: {getattr(field, '@id', '')}, name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
We'll use the record set and field `@id`s noted above.

> **Tip:** You can replace the `record_set_id` below with any valid record set `@id` from the overview step.

In [ ]:
# Extract data from each record set to pandas DataFrames
# Replace the following list with the available record set @id(s) printed above.
record_sets = rs_overview if rs_overview else []
dataframes = {}

for record_set_id in record_sets:
    try:
        print(f"Loading records from record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
        else:
            df = pd.DataFrame()
        dataframes[record_set_id] = df
        print(f"Fields (columns) for record set {record_set_id}:")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Failed to load {record_set_id}: {e}")

# For demonstration, select the first record set as the main (if available)
main_rs_id = record_sets[0] if record_sets else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- Remove outliers or filter on thresholds
- Normalize a chosen numeric field
- Group by a categorical field and show means

**Note:** Update the `numeric_field_id` and `group_field_id` variables with valid column names (`@id`s) from your DataFrame.

In [ ]:
# Use the previously loaded DataFrame for EDA
df = dataframes.get(main_rs_id, pd.DataFrame())
print(f"Shape of main DataFrame: {df.shape}")

# Auto-detect a numeric field for demonstration
import numpy as np
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print("No numeric field found. Please set numeric_field_id manually.")
else:
    print(f"Auto-selected numeric field: {numeric_field_id}")

threshold = 10
if numeric_field_id and numeric_field_id in df:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    norm_name = f"{numeric_field_id}_normalized"
    filtered_df[norm_name] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_name]].head())

    # Auto-detect a non-numeric categorical field to group by
    group_field_id = None
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() < 20:
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
else:
    filtered_df = pd.DataFrame()
    print("No numeric field selected for filtering and normalization.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For demonstration, plot the distribution of the numeric field and the mean by group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
if not filtered_df.empty and numeric_field_id in filtered_df:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

# Plot mean by group if available
if 'group_field_id' in locals() and group_field_id and group_field_id in filtered_df:
    mean_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(8, 4))
    sns.barplot(data=mean_df, x=group_field_id, y=numeric_field_id)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles using `mlcroissant`.

- We loaded and reviewed dataset metadata.
- We listed all available record sets and their field IDs (`@id`).
- We extracted data for one record set and performed basic filtering, normalization, and grouping operations.
- We visualized distributions and group means for a selected numeric field.

This notebook can be extended to deeper biological or statistical analysis as desired by domain experts.
